In [3]:
#1. Preparar el entorno e importar la clase de carga
#El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar src/clases_po 
#y se agrega src a sys.path. Así se puede importar HFDatasetLoader sin instalar el proyecto como paquete.


In [4]:
import sys

print("Versión de Python:", sys.version)
print("Ubicación de Python:", sys.executable)

Versión de Python: 3.14.2 (main, Sep 10 2026, 12:33:04) [GCC 13.3.0]
Ubicación de Python: /workspaces/MachineLearning/.venv/bin/python


In [5]:

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

In [6]:
# 2. Configurar el dataset y la salida

In [7]:
hfdataset = HFDatasetLoader(
    dataset_name="smart-dashcam/motorcycle-accident-driving-datasets",
    split="train",
    filename="datos/Motorcycle-Accident-Driving-Datasets.csv",
)

In [8]:
# 3. Descargar el dataset y crear el DataFrame

In [20]:
df = hfdataset.load()
df.head(2)

,filename,case,duration
0,accident_000.mp4,accident,8.31
1,accident_001.mp4,accident,5.35


In [10]:
# 4. Revisar la estructura del DataFrame

In [11]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

Filas: 136
Columnas: 3
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 136 entries, 0 to 135
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   filename  136 non-null    object 
 1   case      136 non-null    object 
 2   duration  136 non-null    float32
dtypes: float32(1), object(2)
memory usage: 2.8+ KB


### 5. Transformación Mínima de los Datos
Para estandarizar y preparar los datos hacia el modelado:
1. **Renombrado de columnas:** Se traducen a español (`archivo`, `tipo_evento`, `duracion_segundos`).
2. **Filtrado:** Se seleccionan videos con una duración mayor a 5.0 segundos (`duracion_segundos > 5.0`) para conservar eventos con suficiente contexto temporal.

In [23]:
# Aplicar la transformación creando un nuevo DataFrame
df_transformado = df.rename(columns={
    "filename": "archivo",
    "case": "tipo_evento",
    "duration": "duracion_segundos"
})

# Filtrado condicional y reseteo del índice
df_transformado = df_transformado[df_transformado["duracion_segundos"] > 5.0].reset_index(drop=True)

# Verificación de la transformación
print(f"Dimensiones originales:    {df.shape}")
print(f"Dimensiones transformadas: {df_transformado.shape}")
df_transformado.head(10)

Dimensiones originales:    (136, 3)
Dimensiones transformadas: (91, 3)


,archivo,tipo_evento,duracion_segundos
0,accident_000.mp4,accident,8.31
1,accident_001.mp4,accident,5.35
2,accident_0010.mp4,accident,6.71
3,accident_0011.mp4,accident,6.40
4,accident_0012.mp4,accident,7.08
5,accident_0013.mp4,accident,6.65
6,accident_0014.mp4,accident,5.88
7,accident_0018.mp4,accident,7.57
8,accident_0019.mp4,accident,6.40
9,accident_0020.mp4,accident,7.63


### 5. Importar la clase de reportes

In [13]:
from clases_po import Reportes

### 6. Generación del Reporte del DataFrame Transformado
Instanciamos la clase `Reportes` y le pasamos el DataFrame resultante (`df_transformado`) para documentar su estructura en formato de texto.

In [25]:
from clases_po import Reportes

reportes = Reportes()

# Generar y visualizar el reporte del DataFrame transformado
texto = reportes.generar_reporte(df_transformado)
print(texto)

REPORTE DEL DATAFRAME
Filas: 91
Columnas: 3
Duplicados: 0

Columnas y tipos:
- archivo: object
- tipo_evento: object
- duracion_segundos: float32

Valores nulos por columna:
- archivo: 0
- tipo_evento: 0
- duracion_segundos: 0

Estadísticas numéricas:
       duracion_segundos
count             91.000
mean               6.307
std                1.320
min                5.010
25%                5.720
50%                5.720
75%                6.440
max               13.150



### guardar reporte transformado

In [26]:
# Guardar el archivo .txt con el resumen de los datos transformados
ruta = reportes.guardar_reporte(
    df_transformado,
    "reportes/motorcycle_accident_transformed_summary.txt",
)

print(f"Reporte del DataFrame transformado guardado con éxito en: {ruta}")

Reporte del DataFrame transformado guardado con éxito en: reportes/motorcycle_accident_transformed_summary.txt


# Reporte original

In [24]:
reportes = Reportes()
texto = reportes.generar_reporte(df)

print(texto)

REPORTE DEL DATAFRAME
Filas: 136
Columnas: 3
Duplicados: 0

Columnas y tipos:
- filename: object
- case: object
- duration: float32

Valores nulos por columna:
- filename: 0
- case: 0
- duration: 0

Estadísticas numéricas:
       duration
count   136.000
mean      5.417
std       1.754
min       1.470
25%       4.380
50%       5.720
75%       5.783
max      13.150



#### 7. Imprimir el reporte directamente

In [16]:
reportes.imprimir_reporte(df)

REPORTE DEL DATAFRAME
Filas: 136
Columnas: 3
Duplicados: 0

Columnas y tipos:
- filename: object
- case: object
- duration: float32

Valores nulos por columna:
- filename: 0
- case: 0
- duration: 0

Estadísticas numéricas:
       duration
count   136.000
mean      5.417
std       1.754
min       1.470
25%       4.380
50%       5.720
75%       5.783
max      13.150


### 8. Guardar el reporte ORIGINAL en un archivo de texto

In [27]:
ruta = reportes.guardar_reporte(
    df,
    "reportes/motorcycle-accident-driving-datasets-ORIGINAL.txt",
)

print(f"Reporte guardado en: {ruta}")

Reporte guardado en: reportes/motorcycle-accident-driving-datasets-ORIGINAL.txt


CONCLUSIONES 